# Job Submission Behavior Analysis

**Authors:** Jiachen Con, Yanlin Wang

This notebook explores queueing and workload patterns in derived 2025 job-level HPC tables generated by `01_build_master_2025.ipynb`.

Main topics include:

1. Waiting time distribution
2. Submission patterns by weekday and hour of day
3. Runtime distribution for completed jobs
4. Resource request distributions
5. GPU request distribution and GPU job wait times
6. CPU vs GPU waiting time comparison
7. Exploratory workload construction and queue-related modeling
8. Relationships among workload type, runtime, submission timing, and wait time

**Inputs:** `outputs/master_2025_joblevel_submission.csv.gz` and `outputs/master_2025_joblevel_terminal.csv.gz`


In [ ]:
# loading necessary packages
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

# locate project root and derived outputs
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "environment.yml").exists() and (PROJECT_ROOT.parent / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
SUB_PATH = OUTPUT_DIR / "master_2025_joblevel_submission.csv.gz"
TER_PATH = OUTPUT_DIR / "master_2025_joblevel_terminal.csv.gz"

assert SUB_PATH.exists(), f"Missing: {SUB_PATH}"
assert TER_PATH.exists(), f"Missing: {TER_PATH}"

print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"SUB_PATH: {SUB_PATH}")
print(f"TER_PATH: {TER_PATH}")

# load the data
sub = pd.read_csv(SUB_PATH, compression="gzip")
ter = pd.read_csv(TER_PATH, compression="gzip")

# print the name of columns
print("Columns in sub:")
print(sub.columns.tolist())

print("\nColumns in ter:")
print(ter.columns.tolist())

# Count the number of tasks in different states
print(ter["State"].value_counts())


## Data Preprocessing: Convert Timestamps

The raw CSV files store `Submit`, `Start`, and `End` as strings. We convert them to `datetime` objects so that we can:

- Compute time differences (e.g., `Start - Submit`)
- Extract hour / weekday information
- Perform proper time-based analysis

> `errors="coerce"` ensures invalid timestamps become `NaT` instead of raising an exception.

In [ ]:
for df in (sub, ter):
    for c in ["Submit", "Start", "End"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

## Data Preprocessing: Ensure Numeric Types

Columns like `ReqCPUS`, `ReqMem_MB`, and `WaitTimeSec` may be loaded as strings from the CSV. We explicitly cast them to numeric so that:

- Summary statistics (mean, percentiles) are correct
- Histograms and plots render properly
- Arithmetic operations are valid

> `errors="coerce"` converts unparseable entries to `NaN` instead of failing.

In [ ]:
for df in (sub, ter):
    for c in ["ReqCPUS", "ReqNodes", "AllocCPUS", "AllocNodes", "ReqMem_MB", "WaitTimeSec", "RunTimeSec"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

print("submission shape:", sub.shape)
print("terminal shape:  ", ter.shape)

## Q1: Waiting Time Distribution (Start - Submit)

Using the **terminal table** (`master_2025_joblevel_terminal.csv.gz`), which contains `Start`, `End`, and `WaitTimeSec`.

**Excluded:** jobs still in `PENDING` status, and `CANCELLED` jobs that never started (no valid `Start` timestamp).

In [ ]:
if "WaitTimeSec" in ter.columns:
    wait = ter["WaitTimeSec"]
else:
    # compute from timestamps if needed
    wait = (ter["Start"] - ter["Submit"]).dt.total_seconds()

wait = wait.replace([np.inf, -np.inf], np.nan).dropna()
wait_hr = wait / 3600.0

print("\n=== Wait time (hours) summary ===")
print(wait_hr.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

plt.figure()
plt.hist(wait_hr.clip(upper=np.nanpercentile(wait_hr, 99)), bins=60)
plt.xlabel("Wait time (hours) [clipped at 99th pct]")
plt.ylabel("Job count")
plt.title("Wait time distribution (job-level)")
plt.tight_layout()
plt.show()

## Q2: Submission Patterns by Weekday and Hour of Day

Using the **submission table** (`master_2025_joblevel_submission.csv.gz`) to examine when users tend to submit jobs.

In [ ]:
sub["SubmitHour"] = sub["Submit"].dt.hour
sub["SubmitWeekday"] = sub["Submit"].dt.day_name()

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sub["SubmitWeekday"] = pd.Categorical(sub["SubmitWeekday"], categories=weekday_order, ordered=True)

weekday_counts = sub["SubmitWeekday"].value_counts().reindex(weekday_order)
hour_counts = sub["SubmitHour"].value_counts().sort_index()

print("\n=== Submissions by weekday ===")
print(weekday_counts)

print("\n=== Submissions by hour (0-23) ===")
print(hour_counts)

# Chart of Submissions by weekday (e.g., how many submission on monday, etc)
plt.figure()
weekday_counts.plot(kind="bar")
plt.xlabel("Weekday")
plt.ylabel("Submission count")
plt.title("Submissions by weekday")
plt.tight_layout()
plt.show()

# Chart of Submissions by hour of day (e.g., how many submission on 1 A.M., etc)
plt.figure()
hour_counts.plot(kind="bar")
plt.xlabel("Hour of day")
plt.ylabel("Submission count")
plt.title("Submissions by hour of day")
plt.tight_layout()
plt.show()

## Q3: Runtime Distribution (End - Start) for Completed Jobs

Only `COMPLETED` jobs are included, since they have valid `Start` and `End` timestamps. Runtime is reported in **hours**.

In [ ]:
ter["State"] = ter["State"].astype(str).str.upper().str.strip()
completed = ter[ter["State"] == "COMPLETED"].copy()

# Compute runtime (prefer using the precomputed RunTimeSec from the data cleaning stage if available)
if "RunTimeSec" in completed.columns:
    runtime = completed["RunTimeSec"]
else:
    runtime = (completed["End"] - completed["Start"]).dt.total_seconds()

# # Clean the data: remove inf values and set impossible negative runtime values to missing
runtime = runtime.replace([np.inf, -np.inf], np.nan)

neg_cnt = (runtime < 0).sum()
valid_cnt = runtime.notna().sum()
print(f"Negative runtime count: {neg_cnt:,} / {valid_cnt:,} ({neg_cnt/valid_cnt:.6%})")

runtime = runtime.mask(runtime < 0, np.nan).dropna()
runtime_hr = runtime / 3600.0

print("\n=== Runtime (COMPLETED only, hours) summary (after removing negative) ===")
print(runtime_hr.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

plt.figure()
plt.hist(runtime_hr.clip(upper=np.nanpercentile(runtime_hr, 99)), bins=60)
plt.xlabel("Runtime (hours) [clipped at 99th pct]")
plt.ylabel("Job count")
plt.title("Runtime distribution (COMPLETED jobs)")
plt.tight_layout()
plt.show()

## Q4: Resource Request Distributions — ReqCPUS, ReqMem, ReqNodes

Examining the distribution of resources requested per job: number of CPUs, memory (converted to GB), and number of nodes.

In [ ]:
reqcpus = sub["ReqCPUS"].replace([np.inf, -np.inf], np.nan).dropna()
reqnodes = sub["ReqNodes"].replace([np.inf, -np.inf], np.nan).dropna()
reqmem_gb = (sub["ReqMem_MB"].replace([np.inf, -np.inf], np.nan).dropna()) / 1024.0

print("\n=== ReqCPUS summary ===")
print(reqcpus.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\n=== ReqNodes summary ===")
print(reqnodes.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\n=== ReqMem (GB) summary ===")
print(reqmem_gb.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

plt.figure()
plt.hist(reqcpus.clip(upper=np.nanpercentile(reqcpus, 99)), bins=50)
plt.xlabel("ReqCPUS [clipped at 99th pct]")
plt.ylabel("Job count")
plt.title("ReqCPUS distribution")
plt.tight_layout()
plt.show()

plt.figure()
plt.hist(reqnodes.clip(upper=np.nanpercentile(reqnodes, 99)), bins=30)
plt.xlabel("ReqNodes [clipped at 99th pct]")
plt.ylabel("Job count")
plt.title("ReqNodes distribution")
plt.tight_layout()
plt.show()

plt.figure()
plt.hist(reqmem_gb.clip(upper=np.nanpercentile(reqmem_gb, 99)), bins=60)
plt.xlabel("ReqMem (GB) [clipped at 99th pct]")
plt.ylabel("Job count")
plt.title("ReqMem distribution")
plt.tight_layout()
plt.show()

## Q5: GPU Request Distribution

GPU count is extracted from the `ReqTRES` column via regex (e.g., `gres/gpu=2`). This section covers:

1. **GPU request distribution** — how many GPUs each job requests (for GPU jobs only)
2. **GPU job wait time** — waiting time distribution for jobs that require GPUs

In [ ]:
# Regular expression to match patterns like "gres/gpu=1"
gpu_pattern = re.compile(r"gres/gpu=([0-9]+)", re.IGNORECASE)
def extract_gpu_count(tres):
    if pd.isna(tres):
        return 0
    match = gpu_pattern.search(str(tres))
    if match:
        return int(match.group(1))
    return 0
# Create a new column indicating the number of GPUs requested
ter["GPU_Count"] = ter["ReqTRES"].apply(extract_gpu_count)

# Filter GPU jobs from the submission dataset
# A job is considered a GPU job if GPU_Count > 0.
# We create a new DataFrame sub_gpu containing only GPU jobs.
ter_gpu = ter[(ter["GPU_Count"] > 0) & (ter["GPU_Count"] <= 64)].copy()

# GPU request distribution (how many GPUs per job)
gpu_freq = ter_gpu["GPU_Count"].value_counts().sort_index()
print("\n=== GPU_Count frequency table (GPU jobs only) ===")
print(gpu_freq)
plt.figure()
gpu_freq.plot(kind="bar")
plt.xlabel("Number of GPUs requested (GPU_Count)")
plt.ylabel("Job count")
plt.title("Distribution of Requested GPUs per Job (GPU jobs)")
plt.tight_layout()
plt.show()

# Wait time distribution for GPU jobs (in hours)
# Note: WaitTimeSec is computed in your cleaning pipeline.
# We clean invalid values (inf / negative) and drop missing.
wait_gpu = ter_gpu["WaitTimeSec"].replace([np.inf, -np.inf], np.nan)
wait_gpu = wait_gpu.mask(wait_gpu < 0, np.nan).dropna()
wait_gpu_hr = wait_gpu / 3600.0
print("\n=== GPU jobs: Wait time (hours) summary ===")
print(wait_gpu_hr.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
plt.figure()
plt.hist(
    wait_gpu_hr.clip(upper=np.nanpercentile(wait_gpu_hr, 99)),
    bins=60
)
plt.xlabel("Wait time (hours) [clipped at 99th percentile]")
plt.ylabel("Job count")
plt.title("Wait Time Distribution for GPU Jobs")
plt.tight_layout()
plt.show()


## Q6: CPU vs GPU — Is There a Significant Difference in Waiting Time?

**Setup:** Jobs are split into two groups based on `GPU_Count`:
- **GPU jobs:** `GPU_Count > 0`
- **CPU-only jobs:** `GPU_Count == 0`

**Visualization:** Since waiting times are highly right-skewed, we apply `log(1 + wait time)` for histograms and use boxplots (without outliers) for comparison.

**Statistical test:** The Mann-Whitney U test (non-parametric, two-sided) is used because the data are non-normal with extreme values.

In [ ]:
from scipy.stats import mannwhitneyu
# Identify GPU jobs and CPU-only jobs
gpu_jobs = ter[ter["GPU_Count"] > 0].copy()
cpu_jobs = ter[ter["GPU_Count"] == 0].copy()
print("Number of GPU jobs:", len(gpu_jobs))
print("Number of CPU-only jobs:", len(cpu_jobs))

# Extract and clean wait times
wait_gpu = gpu_jobs["WaitTimeSec"].replace([np.inf, -np.inf], np.nan)
wait_gpu = wait_gpu.mask(wait_gpu < 0, np.nan).dropna()
wait_cpu = cpu_jobs["WaitTimeSec"].replace([np.inf, -np.inf], np.nan)
wait_cpu = wait_cpu.mask(wait_cpu < 0, np.nan).dropna()
# convert to hours
wait_gpu_hr = wait_gpu / 3600.0
wait_cpu_hr = wait_cpu / 3600.0

# Summary statistics
print("\n=== GPU wait time summary (hours) ===")
print(wait_gpu_hr.describe(percentiles=[0.5,0.9,0.95,0.99]))
print("\n=== CPU wait time summary (hours) ===")
print(wait_cpu_hr.describe(percentiles=[0.5,0.9,0.95,0.99]))

# Visualization (log scale due to skewness)
plt.figure()
plt.hist(np.log1p(wait_cpu_hr),
         bins=60,
         alpha=0.6,
         label="CPU jobs")
plt.hist(np.log1p(wait_gpu_hr),
         bins=60,
         alpha=0.6,
         label="GPU jobs")
plt.xlabel("log(1 + wait time in hours)")
plt.ylabel("Job count")
plt.title("Wait Time Distribution (CPU vs GPU)")
plt.legend()
plt.tight_layout()
plt.show()

# Boxplot comparison
wait_df = pd.DataFrame({
    "wait_hr": np.concatenate([wait_cpu_hr, wait_gpu_hr]),
    "job_type": ["CPU"]*len(wait_cpu_hr) + ["GPU"]*len(wait_gpu_hr)
})
plt.figure()
wait_df.boxplot(column="wait_hr", by="job_type", showfliers=False)
plt.ylabel("Wait time (hours)")
plt.title("Wait Time Comparison: CPU vs GPU Jobs")
plt.suptitle("")
plt.show()

# Statistical test (Mann–Whitney U)
stat, p_value = mannwhitneyu(wait_cpu_hr,
                             wait_gpu_hr,
                             alternative="two-sided")
print("\nMann–Whitney U test result")
print("Statistic:", stat)
print("p-value:", p_value)

# Interpretation helper
alpha = 0.05
if p_value < alpha:
    print("\nResult: There is a statistically significant difference "
          "in waiting times between CPU-only jobs and GPU jobs.")
else:
    print("\nResult: No statistically significant difference "
          "in waiting times between CPU-only jobs and GPU jobs.")

### Q6 Results

| Metric | GPU Jobs | CPU-only Jobs |
|--------|----------|---------------|
| Count (valid) | 351,138 | 5,166,144 |
| Mean wait (hr) | 8.81 | 6.83 |
| Median wait (hr) | 0.087 | 0.44 |

**Key findings:**

- Both distributions are strongly right-skewed: most jobs start quickly, but a small fraction waits extremely long.
- GPU jobs show a wider spread and more extreme waiting times, suggesting GPU resources are more limited.
- **Mann-Whitney U test:** p-value $\approx$ 0 (far below 0.05). We reject the null hypothesis — there is a **statistically significant difference** in waiting times between CPU-only and GPU jobs.
- This is likely due to the higher demand and limited availability of GPU resources in the cluster.

## Q7: Predicting GPU Job Waiting Time

**Model used:**
- Random Forest
- XGBoost

The empirical result shows that the Random Forest model has higher prediction accuracy.

**Features used:**
- Categorical: `SubmitWeekday`, `SubmitHour`, `QOS`
- Numeric: `GPU_Count`, `ReqCPUS`, `ReqMem_MB`, `Priority`

In [ ]:
# Q7: Direct regression (No grouping) + Model comparison

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# Winsorize functions
def winsorize_df(df, cols, lower=0.025, upper=0.975):
    df = df.copy()
    bounds = {}
    for c in cols:
        low = df[c].quantile(lower)
        high = df[c].quantile(upper)
        df[c] = np.clip(df[c], low, high)
        bounds[c] = (low, high)
    return df, bounds

def apply_winsorize(df, bounds):
    df = df.copy()
    for c, (low, high) in bounds.items():
        df[c] = np.clip(df[c], low, high)
    return df

# Data cleaning
model_gpu = ter_gpu.copy()
model_gpu["WaitTimeSec"] = pd.to_numeric(model_gpu["WaitTimeSec"], errors="coerce")
model_gpu = model_gpu.replace([np.inf, -np.inf], np.nan)
model_gpu = model_gpu[model_gpu["WaitTimeSec"].notna()]
model_gpu = model_gpu[model_gpu["WaitTimeSec"] >= 0].copy()
model_gpu["WaitTimeHr"] = model_gpu["WaitTimeSec"] / 3600.0
model_gpu = model_gpu[model_gpu["WaitTimeHr"] >= 0]
model_gpu["Submit"] = pd.to_datetime(model_gpu["Submit"], errors="coerce")
model_gpu["SubmitHour"] = model_gpu["Submit"].dt.hour
model_gpu["SubmitWeekday"] = model_gpu["Submit"].dt.day_name()

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
model_gpu["SubmitWeekday"] = pd.Categorical(
    model_gpu["SubmitWeekday"],
    categories=weekday_order,
    ordered=True
)

# Features
features_cat = ["SubmitWeekday","QOS","SubmitHour"]
features_num = ["GPU_Count","ReqCPUS", "ReqMem_MB", "Priority"]
use_cols = features_cat + features_num + ["WaitTimeHr"]
model_gpu = model_gpu[use_cols].copy()
X = model_gpu[features_cat + features_num]
y = model_gpu["WaitTimeHr"]

sample_n = min(200000, len(model_gpu))
sample_idx = model_gpu.sample(n=sample_n, random_state=SEED).index
X = X.loc[sample_idx]
y = y.loc[sample_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED
)

X_train[features_num], win_bounds = winsorize_df(X_train[features_num], features_num)
X_test[features_num] = apply_winsorize(X_test[features_num], win_bounds)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, features_num),
        ("cat", categorical_transformer, features_cat)
    ]
)
y_train_log = np.log1p(y_train)

# Model 1: XGBoost
xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squaredlogerror",
        random_state=SEED
    ))
])

xgb_model.fit(X_train, y_train_log)

pred_log_xgb = xgb_model.predict(X_test)
pred_xgb = np.maximum(np.expm1(pred_log_xgb), 0)


# Model 2: Random Forest
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=SEED,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train_log)

pred_log_rf = rf_model.predict(X_test)
pred_rf  = np.maximum(np.expm1(pred_log_rf), 0)

# Evaluation
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n=== {name} ===")
    print(f"MAE  = {mae:.4f} hours")
    print(f"RMSE = {rmse:.4f} hours")

evaluate(y_test, pred_xgb, "XGBoost")
evaluate(y_test, pred_rf, "Random Forest")

#### Result Visualization

In [ ]:
# RMSLE comparison (XGB vs RF)
from sklearn.metrics import mean_squared_log_error
import numpy as np
import matplotlib.pyplot as plt

# XGBoost
y_true_xgb = y_test.clip(lower=0)
y_pred_xgb = pd.Series(pred_xgb, index=y_test.index).clip(lower=0)

mask_xgb = (~y_true_xgb.isna()) & (~y_pred_xgb.isna())
y_true_xgb_clean = y_true_xgb[mask_xgb]
y_pred_xgb_clean = y_pred_xgb[mask_xgb]

rmsle_xgb = np.sqrt(mean_squared_log_error(y_true_xgb_clean, y_pred_xgb_clean))


# Random Forest
y_true_rf = y_test.clip(lower=0)
y_pred_rf = pd.Series(pred_rf, index=y_test.index).clip(lower=0)

mask_rf = (~y_true_rf.isna()) & (~y_pred_rf.isna())
y_true_rf_clean = y_true_rf[mask_rf]
y_pred_rf_clean = y_pred_rf[mask_rf]

rmsle_rf = np.sqrt(mean_squared_log_error(y_true_rf_clean, y_pred_rf_clean))

# Print results
print("\n RMSLE Comparison ")
print(f"XGBoost RMSLE = {rmsle_xgb:.4f}")
print(f"Random Forest RMSLE = {rmsle_rf:.4f}")

plt.figure(figsize=(6,6))

# XGB
log_true_xgb = np.log1p(y_true_xgb_clean)
log_pred_xgb = np.log1p(y_pred_xgb_clean)

plt.scatter(log_true_xgb, log_pred_xgb, alpha=0.3, s=10, label="XGBoost")

# RF
log_true_rf = np.log1p(y_true_rf_clean)
log_pred_rf = np.log1p(y_pred_rf_clean)

plt.scatter(log_true_rf, log_pred_rf, alpha=0.3, s=10, label="Random Forest")

# y = x line
min_val = min(log_true_xgb.min(), log_pred_xgb.min(), log_true_rf.min(), log_pred_rf.min())
max_val = max(log_true_xgb.max(), log_pred_xgb.max(), log_true_rf.max(), log_pred_rf.max())

plt.plot([min_val, max_val], [min_val, max_val], 'r--')

plt.xlabel("log(True Wait Time)")
plt.ylabel("log(Predicted Wait Time)")
plt.title("Log-Scale Prediction Comparison")

plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)

plt.legend()
plt.tight_layout()
plt.show()

## Q8: Workload Variable Construction (log-transform + K-Means)

We construct a single ordinal variable **`workload`** that captures overall job resource intensity by clustering three features: `ReqCPUS`, `GPU_Count`, and `ReqMem_MB`.

**Why log-transform?** All three features are heavily right-skewed (e.g., `ReqMem_MB` ranges from ~1 MB to 500+ GB). A raw `StandardScaler` would let extreme outliers dominate clustering. Applying `log(1+x)` first compresses the long tail and produces a more balanced feature space.

**Why exclude `ReqNodes`?** Over 99% of jobs request exactly 1 node (see Q4 results: the 95th percentile is still 1). A near-constant feature adds no discriminative power and only introduces noise.

**Pipeline:**
1. `log(1+x)` transform on all three features
2. `StandardScaler` to zero-mean, unit-variance
3. Elbow Method to choose *k*
4. K-Means clustering, with semantic labels (Light / Medium / Heavy / ...) assigned by ranking each cluster's average resource usage

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# ---------- 1. Prepare features (ReqNodes excluded — nearly constant at 1) ----------
workload_features = ["ReqCPUS", "GPU_Count", "ReqMem_MB"]

wl_data = ter[workload_features].copy()
valid_mask = (
    wl_data.notna().all(axis=1)
    & np.isfinite(wl_data.to_numpy()).all(axis=1)
    & (wl_data["GPU_Count"] <= 64)
)
wl_valid = wl_data.loc[valid_mask].copy()

print(f"Valid rows for clustering: {len(wl_valid):,} / {len(ter):,}")

# Standardize RAW features (no log)
scaler_raw = StandardScaler()
wl_raw_scaled = scaler_raw.fit_transform(wl_valid)

# log(1+x) then standardize
wl_log = np.log1p(wl_valid)
scaler = StandardScaler()
wl_scaled = scaler.fit_transform(wl_log)

# Sample for visualisation & clustering
SAMPLE_N = 50_000
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(wl_scaled), size=min(SAMPLE_N, len(wl_scaled)), replace=False)

wl_sample_raw = wl_raw_scaled[sample_idx]
wl_sample = wl_scaled[sample_idx]

# ---------- 2. PCA comparison: raw vs log(1+x) ----------
pca_raw = PCA(n_components=2)
pc_raw = pca_raw.fit_transform(wl_sample_raw)

pca_log = PCA(n_components=2)
pc_log = pca_log.fit_transform(wl_sample)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(pc_raw[:, 0], pc_raw[:, 1], s=2, alpha=0.3)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("PCA projection of raw standardized features")

axes[1].scatter(pc_log[:, 0], pc_log[:, 1], s=2, alpha=0.3)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("PCA projection of log(1+x) standardized features")

plt.tight_layout()
plt.show()

Even after log transformation, the feature space exhibits clear structured patterns rather than a diffuse cloud. This reflects the fact that resource requests are **not continuously distributed**, but instead concentrated on a limited set of **standard configurations** (e.g., fixed CPU counts, discrete GPU allocations, and predefined memory sizes).

In [ ]:
# ---------- 3. Choose k using Elbow Method ----------
K_range = range(2, 10)
inertias = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    km.fit(wl_sample)
    inertias.append(km.inertia_)
    print(f"k={k}  inertia={km.inertia_:.0f}")

# ---------- Plot ----------
plt.figure(figsize=(6, 4))
plt.plot(list(K_range), inertias, "o-")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.tight_layout()
plt.show()

We use the elbow method to determine the number of clusters.
The inertia decreases rapidly from k = 2 to k = 3, and then the rate of decrease slows down significantly.
This indicates diminishing returns in increasing k beyond 3.

Therefore, we choose k = 3 as a balance between model complexity and interpretability.

In [ ]:
CHOSEN_K = 3

km_final = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=20)
km_final.fit(wl_sample)

all_labels = km_final.predict(wl_scaled)

cluster_df = wl_valid.copy()
cluster_df["cluster"] = all_labels

centers_scaled = km_final.cluster_centers_
cluster_scores = centers_scaled.mean(axis=1)
rank_order = np.argsort(cluster_scores).tolist()

label_names = ["Light", "Medium", "Heavy"]
cluster_to_label = {cid: label_names[i] for i, cid in enumerate(rank_order)}

cluster_df["workload"] = pd.Categorical(
    pd.Series(all_labels).map(cluster_to_label),
    categories=label_names,
    ordered=True
)

print("=== Workload category counts ===")
print(cluster_df["workload"].value_counts().reindex(label_names))

# ---------- Workload distribution ----------
plt.figure(figsize=(9, 6))

cluster_df["workload"].value_counts().reindex(label_names).plot(kind="bar")

plt.xlabel("Workload Category")
plt.ylabel("Job Count")
plt.title("Distribution of Workload Categories")

plt.tight_layout()
plt.show()


print("\n=== Mean resource usage per workload category ===")

summary = cluster_df.groupby("workload", observed=False)[workload_features].mean().reindex(label_names)
summary["ReqMem_GB"] = summary["ReqMem_MB"] / 1024
summary_display = summary.drop(columns=["ReqMem_MB"])
summary_display = summary_display.round(3)
print(summary_display)

# ---------- PCA scatter colored by workload cluster ----------
sample_labels = km_final.predict(wl_sample)
sample_label_names = [cluster_to_label[c] for c in sample_labels]

fig, ax = plt.subplots(figsize=(9, 6))

colors = plt.cm.tab10.colors
for i, name in enumerate(label_names):
    mask = [n == name for n in sample_label_names]
    ax.scatter(pc_log[mask, 0], pc_log[mask, 1],
               s=3, alpha=0.4, color=colors[i], label=name)

# Mark cluster centers in PCA space
centers_pc = pca_log.transform(km_final.cluster_centers_)
for cid, (cx, cy) in enumerate(centers_pc):
    ax.scatter(cx, cy, marker="X", s=200, edgecolors="black",
               linewidths=1.2, color=colors[label_names.index(cluster_to_label[cid])])

ax.set_xlabel(f"PC1")
ax.set_ylabel(f"PC2")
ax.set_title("Workload Clusters in PCA Space")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()



We apply K-means clustering (k = 3) on log-transformed and standardized resource requests (CPU, GPU, memory) to construct a workload variable.

The resulting clusters can be interpreted as:
1. **Light**: low CPU, low memory, no GPU
   
2. **Medium**: CPU- and memory-intensive jobs

3. **Heavy**: GPU-intensive and high-memory jobs

## Q9: CPU-only job waiting time prediction
**Model used:**
- Random Forest

**Features used:**
- Categorical: `SubmitWeekday`, `SubmitHour`, `QOS`
- Numeric: `GPU_Count`, `ReqCPUS`, `ReqMem_MB`, `Priority`

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
SEED = 42
np.random.seed(SEED)
def winsorize_df(df, cols, lower=0.025, upper=0.975):
    df = df.copy()
    bounds = {}
    for c in cols:
        low = df[c].quantile(lower)
        high = df[c].quantile(upper)
        df[c] = np.clip(df[c], low, high)
        bounds[c] = (low, high)
    return df, bounds

def apply_winsorize(df, bounds):
    df = df.copy()
    for c, (low, high) in bounds.items():
        df[c] = np.clip(df[c], low, high)
    return df
model_cpu = df[df["GPU_Count"] == 0].copy()

model_cpu["WaitTimeSec"] = pd.to_numeric(model_cpu["WaitTimeSec"], errors="coerce")
model_cpu = model_cpu.replace([np.inf, -np.inf], np.nan)
model_cpu = model_cpu[model_cpu["WaitTimeSec"].notna()]
model_cpu = model_cpu[model_cpu["WaitTimeSec"] >= 0].copy()

model_cpu["WaitTimeHr"] = model_cpu["WaitTimeSec"] / 3600.0
model_cpu = model_cpu[model_cpu["WaitTimeHr"] >= 0]

model_cpu["Submit"] = pd.to_datetime(model_cpu["Submit"], errors="coerce")
model_cpu["SubmitHour"] = model_cpu["Submit"].dt.hour
model_cpu["SubmitWeekday"] = model_cpu["Submit"].dt.day_name()

weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
model_cpu["SubmitWeekday"] = pd.Categorical(
    model_cpu["SubmitWeekday"],
    categories=weekday_order,
    ordered=True
)
features_cat = ["SubmitWeekday","QOS","SubmitHour"]
features_num = ["GPU_Count","ReqCPUS", "ReqMem_MB", "Priority"]
use_cols = features_cat + features_num + ["WaitTimeHr"]
model_cpu = model_cpu[use_cols].copy()
X = model_cpu[features_cat + features_num]
y = model_cpu["WaitTimeHr"]
sample_n = min(200000, len(model_cpu))
sample_idx = model_cpu.sample(n=sample_n, random_state=SEED).index
X = X.loc[sample_idx]
y = y.loc[sample_idx]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED
)
X_train[features_num], win_bounds = winsorize_df(X_train[features_num], features_num)
X_test[features_num] = apply_winsorize(X_test[features_num], win_bounds)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, features_num),
        ("cat", categorical_transformer, features_cat)
    ]
)
y_train_log = np.log1p(y_train)
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=SEED,
        n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train_log)
pred_log_rf = rf_model.predict(X_test)
pred_rf = np.maximum(np.expm1(pred_log_rf), 0)
mae = mean_absolute_error(y_test, pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, pred_rf))
rmsle = np.sqrt(np.mean((np.log1p(pred_rf) - np.log1p(y_test))**2))

print("\n Random Forest ")
print(f"MAE  = {mae:.4f} hours")
print(f"RMSE = {rmse:.4f} hours")

print("\n RMSLE Comparison ")
print(f"Random Forest RMSLE = {rmsle:.4f}")

#### Result Visualization

In [ ]:
from sklearn.metrics import mean_squared_log_error
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

y_true_rf = y_test.clip(lower=0)
y_pred_rf = pd.Series(pred_rf, index=y_test.index).clip(lower=0)
mask_rf = (~y_true_rf.isna()) & (~y_pred_rf.isna())
y_true_rf_clean = y_true_rf[mask_rf]
y_pred_rf_clean = y_pred_rf[mask_rf]
rmsle_rf = np.sqrt(mean_squared_log_error(y_true_rf_clean, y_pred_rf_clean))
print("\n RMSLE Comparison ")
print(f"Random Forest RMSLE = {rmsle_rf:.4f}")
plt.figure(figsize=(6,6))
log_true_rf = np.log1p(y_true_rf_clean)
log_pred_rf = np.log1p(y_pred_rf_clean)
plt.scatter(log_true_rf, log_pred_rf, alpha=0.3, s=10, label="Random Forest")
min_val = min(log_true_rf.min(), log_pred_rf.min())
max_val = max(log_true_rf.max(), log_pred_rf.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.xlabel("log(True Wait Time)")
plt.ylabel("log(Predicted Wait Time)")
plt.title("Log-Scale Prediction Comparison (CPU)")
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.legend()
plt.tight_layout()
plt.show()

## Q10: Workload vs. Wait Time Analysis (Distribution + CDF)

We analyze how job waiting time varies across the constructed workload categories (Light / Medium / Heavy). 

In [ ]:
cluster_df = cluster_df.copy()

cluster_df["WaitTimeSec"] = (
    ter.loc[valid_mask, "WaitTimeSec"]
    if "WaitTimeSec" in ter.columns
    else (ter.loc[valid_mask, "Start"] - ter.loc[valid_mask, "Submit"]).dt.total_seconds()
)

cluster_df["WaitTimeSec"] = cluster_df["WaitTimeSec"].replace([np.inf, -np.inf], np.nan)
cluster_df = cluster_df.dropna(subset=["WaitTimeSec"])

cluster_df["WaitTime_hr"] = cluster_df["WaitTimeSec"] / 3600.0

# summary stats
wait_median = (
    cluster_df.groupby("workload", observed=False)["WaitTime_hr"]
    .median()
    .reindex(label_names)
)

wait_mean = (
    cluster_df.groupby("workload", observed=False)["WaitTime_hr"]
    .mean()
    .reindex(label_names)
)

# side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

wait_median.plot(kind="bar", ax=axes[0])
axes[0].set_title("Median Wait Time by Workload")
axes[0].set_ylabel("Hours")
axes[0].set_xlabel("Workload")
axes[0].tick_params(axis="x", rotation=0)

wait_mean.plot(kind="bar", ax=axes[1])
axes[1].set_title("Mean Wait Time by Workload")
axes[1].set_ylabel("Hours")
axes[1].set_xlabel("Workload")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

wait_summary = pd.DataFrame({
    "Median Wait Time (hr)": wait_median.round(2),
    "Mean Wait Time (hr)": wait_mean.round(2)
})

print(wait_summary)

Interestingly, we observe that **GPU-intensive workloads (“Heavy”) exhibit the shortest median wait time**, indicating that most such jobs are scheduled quickly. This can be attributed to the relatively smaller number of GPU jobs and the use of dedicated GPU resources, which results in lower contention compared to CPU-based workloads.

However, despite their short median wait time, **GPU workloads also have the highest average wait time**, suggesting the presence of a heavy right tail in their distribution. In contrast, lightweight jobs tend to experience longer typical wait times due to the high volume of submissions competing for shared CPU resources.

In [ ]:
import seaborn as sns

plt.figure(figsize=(7,5))
sns.boxplot(
    data=cluster_df,
    x="workload",
    y="WaitTime_hr",
    showfliers=False
)
plt.ylim(0, cluster_df["WaitTime_hr"].quantile(0.95)) 
plt.title("Wait Time Distribution by Workload")
plt.show()

We observe that **Heavy workloads exhibit substantially greater variability**, with a wider interquartile range and longer upper tails compared to Light and Medium workloads. This indicates that **GPU-intensive jobs experience more heterogeneous scheduling outcomes**. In contrast, **Light and Medium workloads show relatively more concentrated distributions**, suggesting more stable and predictable wait times.

In [ ]:
plt.figure(figsize=(8,5))

for wl in label_names:
    data = cluster_df[cluster_df["workload"] == wl]["WaitTime_hr"]
    data = data[data <= data.quantile(0.99)] +1e-6 
    data = np.sort(data)
    
    y = np.arange(len(data)) / len(data)
    plt.plot(data, y, label=wl)

plt.xscale("log")
plt.xlabel("Wait Time (hours, log scale)")
plt.ylabel("CDF")
plt.title("CDF of Wait Time by Workload")
plt.legend()

plt.tight_layout()
plt.show()

The CDF reveals a clear behavior for workloads.

At very short wait times, Heavy workloads dominate, indicating that a large proportion of GPU jobs are scheduled almost immediately. This explains the low median wait time observed earlier.

However, as wait time increases, the CDF curve for Heavy workloads rises more slowly than those of Light and Medium workloads, indicating that a subset of GPU jobs experiences significantly longer delays.

Overall, these results reveal a clear two-phase behavior for GPU-intensive workloads: most jobs are scheduled quickly, while a smaller subset experiences substantial waiting times. This dual behavior explains the observed discrepancy between median and mean wait times.

## Q11: Workload vs. Run Time (Execution Characteristics)

In [ ]:
completed_mask = ter["State"].str.upper().str.strip() == "COMPLETED"

cluster_df = cluster_df.copy()

cluster_df["RunTimeSec"] = (
    ter.loc[valid_mask, "RunTimeSec"]
    if "RunTimeSec" in ter.columns
    else (ter.loc[valid_mask, "End"] - ter.loc[valid_mask, "Start"]).dt.total_seconds()
)


cluster_df = cluster_df.loc[completed_mask.loc[valid_mask]]


cluster_df["RunTimeSec"] = cluster_df["RunTimeSec"].replace([np.inf, -np.inf], np.nan)
cluster_df = cluster_df.dropna(subset=["RunTimeSec"])
cluster_df = cluster_df[cluster_df["RunTimeSec"] >= 0]

cluster_df["RunTime_hr"] = cluster_df["RunTimeSec"] / 3600.0

run_median = cluster_df.groupby("workload")["RunTime_hr"].median().reindex(label_names)
run_mean = cluster_df.groupby("workload")["RunTime_hr"].mean().reindex(label_names)

summary = pd.DataFrame({
    "Median Run Time (hr)": run_median.round(2),
    "Mean Run Time (hr)": run_mean.round(2)
})

print(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

run_median.plot(kind="bar", ax=axes[0])
axes[0].set_title("Median Run Time by Workload")
axes[0].set_ylabel("Hours")
axes[0].tick_params(axis="x", rotation=0)

run_mean.plot(kind="bar", ax=axes[1])
axes[1].set_title("Mean Run Time by Workload")
axes[1].set_ylabel("Hours")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(7,5))

sns.boxplot(
    data=cluster_df,
    x="workload",
    y="RunTime_hr",
    showfliers=False
)

plt.yscale("log")
plt.title("Run Time Distribution by Workload (log scale)")
plt.ylabel("Run Time (hours)")
plt.show()

We observe that **Heavy workloads exhibit longer run times, both in terms of median and mean**, reflecting their higher computational demands. In contrast, Light and Medium workloads show very similar run time distributions, suggesting **limited differentiation within CPU-based jobs**.

Interestingly, when compared with wait time (see ## Q9), a contrasting pattern emerges: although Heavy workloads take longer to execute, they have shorter median wait times.

This indicates that execution duration alone does not determine scheduling delay. Instead, wait time is more strongly influenced by resource availability and contention, particularly for specialized resources such as GPUs.

## Q12: Wait time vs. Submission time (hour-of-day / weekday effects)

In [ ]:
cluster_df = cluster_df.copy()

cluster_df["SubmitHour"] = ter.loc[valid_mask, "Submit"].dt.hour

wait_by_hour_median = cluster_df.groupby("SubmitHour")["WaitTime_hr"].median()
wait_by_hour_mean = cluster_df.groupby("SubmitHour")["WaitTime_hr"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

wait_by_hour_median.plot(ax=axes[0])
axes[0].set_title("Median Wait Time by Hour")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Hours")

wait_by_hour_mean.plot(ax=axes[1])
axes[1].set_title("Mean Wait Time by Hour")
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("Hours")

plt.tight_layout()
plt.show()

In [ ]:
cluster_df["SubmitWeekday"] = ter.loc[valid_mask, "Submit"].dt.day_name()

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cluster_df["SubmitWeekday"] = pd.Categorical(
    cluster_df["SubmitWeekday"],
    categories=weekday_order,
    ordered=True
)

wait_by_weekday_median = cluster_df.groupby("SubmitWeekday")["WaitTime_hr"].median()
wait_by_weekday_mean = cluster_df.groupby("SubmitWeekday")["WaitTime_hr"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

wait_by_weekday_median.plot(kind="bar", ax=axes[0])
axes[0].set_title("Median Wait Time by Weekday")
axes[0].set_ylabel("Hours")

wait_by_weekday_mean.plot(kind="bar", ax=axes[1])
axes[1].set_title("Mean Wait Time by Weekday")
axes[1].set_ylabel("Hours")

plt.tight_layout()
plt.show()

Temporal analysis shows moderate variation across hours and weekdays, with noticeable spikes in mean wait time driven by extreme delays.

However, the median remains relatively stable, indicating that typical jobs experience similar waiting times regardless of submission time.

This suggests that: **Submission time has only a limited effect on scheduling delay.**

## Q13: WaitTime vs. Run Time 

In [ ]:
sns.scatterplot(
    data=cluster_df.sample(50000),
    x="RunTime_hr",
    y="WaitTime_hr",
    alpha=0.3
)
plt.xscale("log")
plt.xlabel("Run Time (hours, log scale)")
plt.yscale("log")
plt.ylabel("Wait Time (hours, log scale)")
plt.show()

In [ ]:
from scipy.stats import spearmanr

df = cluster_df.copy()

df = df[(df["RunTime_hr"] > 0) & (df["WaitTime_hr"] > 0)]

rt_cap = df["RunTime_hr"].quantile(0.99)
wt_cap = df["WaitTime_hr"].quantile(0.99)

df = df[
    (df["RunTime_hr"] <= rt_cap) &
    (df["WaitTime_hr"] <= wt_cap)
]

sample_df = df.sample(50000, random_state=42)

corr, pval = spearmanr(sample_df["RunTime_hr"], sample_df["WaitTime_hr"])

print(f"Spearman correlation: {corr:.4f}, p-value: {pval:.2e}")

A scatter plot (on log-log scale) shows **no clear pattern**, and this is confirmed by **a low Spearman correlation coefficient**, indicating only a weak monotonic relationship.

This suggests that: **Jobs that run longer are not necessarily scheduled later.**
